# Transfer Learning on Cats vs Dogs with HuggingFace ViT

## Feature Extraction

### CIML Summer Institute
### UC San Diego

This notebook uses Hugging Face Transformers to solve an image classification problem with a pre-trained Vision Transformer (ViT).
We load the model with `AutoModelForImageClassification.from_pretrained(...)`, which gives us a backbone plus a classifier head, and then we freeze most of the backbone so only the final layer learns the cats-vs-dogs decision boundary.

That type of transfer learning is referred to as feature extraction. It is useful because the model can reuse visual features learned on a large pretraining corpus, so training is faster and usually less data-hungry than fitting everything from scratch.

For the image inputs, `ViTImageProcessor.from_pretrained(...)` gives us the preprocessing recipe associated with the checkpoint, including resizing and normalization. Matching the processor to the checkpoint matters because the backbone expects inputs in the same format it saw during pretraining.

Base model:
https://huggingface.co/google/vit-base-patch16-224-in21k

`AutoModelForImageClassification` docs:
https://huggingface.co/docs/transformers/en/model_doc/auto#transformers.AutoModelForImageClassification

In the finetuning notebook, you will see the same Hugging Face workflow, but with data augmentation and a different freezing strategy so more of the backbone can adapt to the new task.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch


# HuggingFace imports
from datasets import load_dataset
from transformers import DefaultDataCollator
from transformers import AutoModelForImageClassification
from transformers import TrainingArguments, Trainer
from transformers import pipeline
from transformers import EarlyStoppingCallback

# Evaluation
import evaluate
from sklearn.metrics import classification_report

plt.rcParams["figure.facecolor"] = "white"

## Define Parameters

These values control the experiment budget and make the run reproducible.
The checkpoint expects 224x224 inputs, the batch size affects memory use, and the seed helps keep the Hugging Face / PyTorch pipeline stable across runs.


In [ ]:
# Input shape/statistics expected by ViT family checkpoints.
IMAGE_DIM = 224
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

# Core training hyperparameters for linear probing.
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3
N_GPUS = 1
N_CPUS = int(os.environ.get("SLURM_CPUS_ON_NODE", 1))

seed = 42

# Data location
DATA_DIR = os.environ.get("CIML26_DATA_DIR") + "/catsVsDogs"

# Where checkpoints and logs are written.
OUTPUT_DIR = "vit_cats_dogs_model/feature_extraction"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device {device} with {N_GPUS} GPUs and {N_CPUS} CPUs.")

In [ ]:
!jupyter --version
print (torch.__version__)
!python --version

# TODO:  Use linux shell command nvidia-smi to see GPU device.  
==> YOUR CODE HERE

## Load Data

We use `datasets.load_dataset("imagefolder", ...)` because it turns a folder structure on disk into a supervised dataset with train, validation, and test splits.
That gives us a clean way to connect image files to labels without writing custom data-loading code.

Checking the split sizes before training is a good habit: if the counts are wrong, the issue is usually in the directory layout or in the dataset path, not in the model itself.

In [ ]:
# Load dataset
data = load_dataset("imagefolder", data_dir=DATA_DIR)

# Print dataset information
print(f"Train dataset size: {len(data['train'])}")
print(f"Validation dataset size: {len(data['validation'])}")

# TODO: Print information about the test split length.
# HINT: Mirror the print statement above using data['test'].
==> YOUR CODE HERE

## Image Preprocessing

`ViTImageProcessor.from_pretrained(...)` loads the resizing and normalization recipe that matches the pre-trained checkpoint we are using.
That matters because the backbone was pretrained on inputs with a particular scale and format, so the model performs best when the new images are transformed the same way.

In feature extraction, preprocessing stays deterministic. We want the classifier head to learn from stable features rather than from random augmentations of the same image.

In [ ]:
from transformers import ViTImageProcessor

# Use the processor attached to the exact checkpoint we train/evaluate.
image_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

def preprocess(examples):
    processed = image_processor(examples["image"], return_tensors="pt")
    pixel_values = processed["pixel_values"].squeeze(0)  # (3, 224, 224)
    return {
        "pixel_values": pixel_values,
        "label": examples["label"],
    }
    

In [ ]:
# Set seed for deterministic multiprocessing
from tranformers import set_seed
set_seed(seed)

# Apply preprocessing to all datasets with multiprocessing and batching
processed_data = {}

for split_name, split_data in data.items():
    processed_data[split_name] = split_data.map(
        preprocess_batch,
        remove_columns=['image'],       # Remove original image column
        desc=f"Processing {split_name}"
    )

# Tell datasets to return torch tensors for these columns.
for split in processed_data:
    processed_data[split].set_format(type="torch", columns=["pixel_values", "label"])

## Set Up Model

`AutoModelForImageClassification` gives us a pre-trained backbone plus a task-specific classification head in one class.
The backbone turns each image into a feature vector, and the classifier head learns the cats-vs-dogs decision boundary on top of those frozen features.

This notebook uses linear probing, so the key question is: how much can we learn by training only the small head instead of the full network?

In [ ]:
# TODO: Complete this cell

# Step 1: Build label lookup dictionaries used in metrics/inference.
labels = data["train"].features["label"].names
id2label = {i: label for i, label in enumerate(labels)}

# TODO: Create a reverse mapping from labels to id number
# Hint use the id2label mapping defined above to create the reverse mapping
label2id = ==> YOUR CODE HERE

# Step 2: Load the pre-trained ViT classification model.
model = AutoModelForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # Replace ImageNet classifier with 2-class head.
)

# Step 3: Freeze the ViT backbone for feature extraction (classifier only trains).
# HINT: Set param.requires_grad = False
for name, param in model.named_parameters():
    if "classifier" not in name:
        ==> YOUR CODE HERE

# Step 4: Print model architecture summary 
print(model)
total_params = sum(p.numel() for p in model.parameters())

# TODO: Count number of trainable parameters in the model
# HINT: Code with be similar to counting total params but use a clause to only count if p.requires_grad is true
trainable_params = ==> YOUR CODE HERE

# TODO: Compute frozen parameters from total - trainable.
frozen_params = ==> YOUR CODE HERE

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params:.2%})")
print(f"Frozen parameters: {frozen_params:,} ({frozen_params/total_params:.2%})")

print(f"Number of output neurons: {model.classifier.out_features}")

## Set Up Training

`Trainer` handles the repetitive parts of training: batching, backpropagation, evaluation, checkpointing, and metric logging.
It works together with `TrainingArguments`, `DefaultDataCollator`, and `EarlyStoppingCallback`.

Because only the head is trainable, the main thing to watch is whether validation accuracy improves quickly and then levels off.

In [ ]:
# Define metrics for evaluation
metric = evaluate.load("accuracy")

# Trainer calls this during eval/predict and logs returned metrics.
def compute_metrics(eval_pred):
    """Convert logits to predicted class ids and compute accuracy."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# Configure training loop, checkpointing, and evaluation cadence.
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    # Runtime optimizations.
    bf16=True,
    dataloader_num_workers=N_CPUS,

    push_to_hub=False,
    report_to="none",
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]

# Default collator stacks tensors from dict-based dataset items.
data_collator = DefaultDataCollator()

In [ ]:
# TODO: Initialize the Trainer with the same components defined above.
# HINT: Wire in model, training_args, processed train/validation splits, collator, metrics, and callbacks.

trainer = Trainer(
    model= ==> YOUR CODE HERE,
    args= ==> YOUR CODE HERE,
    train_dataset=processed_data['train'],
    eval_dataset= ==> YOUR CODE HERE,
    data_collator= ==> YOUR CODE HERE,
    compute_metrics= ==> YOUR CODE HERE,
    callbacks= ==> YOUR CODE HERE,
    )

## Train Model

Since the backbone is frozen, training should be much faster than full finetuning.
Use the validation metric to judge whether the classifier head is actually learning a useful boundary, rather than relying only on the training loss.

In [ ]:
# Train the model
train_results = trainer.train()

# Print basic training metrics
print(f"Training results: {train_results}")

# Save the model
trainer.save_model("vit_cats_dogs_model/feature_extraction_best_model")

## Evaluate Model

Accuracy is a useful summary, but the classification report shows precision and recall for each class.
That extra detail helps you see whether the frozen representation separates cats and dogs evenly or whether the model is favoring one label.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Use trainer.predict() to get predictions for each split
for split_key in processed_data.keys():
    # Get predictions and labels
    predictions = trainer.predict(test_dataset=processed_data[split_key])
    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids
    
    # Generate sklearn classification report
    print(f"{split_key.capitalize()}:")
    print(classification_report(y_true, y_pred, digits=4))
    print()  # Add blank line between splits

## Inference

A Hugging Face pipeline bundles preprocessing, model execution, and label decoding into one object.

In [ ]:
# Create an inference pipeline
classifier = pipeline(
    "image-classification",
    model=model,
    image_processor=image_processor,
)

from PIL import Image

def predict_image(image_path):
    """Predict class probabilities for one image and visualize the result."""
    image = Image.open(image_path)
    result = classifier(image)

    # Visual sanity-check: does predicted label match what you see?
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"Prediction: {result[0]['label']} ({result[0]['score']:.2f})")
    plt.axis("off")
    plt.show()

    return result

In [ ]:
# Perform inference on a single test image
image_path = DATA_DIR + "/test/cats/cat.1070.jpg"
predict_image(image_path)

In [ ]:
# TODO: Perform inference on this image using predict_image()
image_path = DATA_DIR + "/test/dogs/dog.1233.jpg"
==> YOUR CODE HERE

# TODO: Perform inference on this test image using predict_image()
image_path = DATA_DIR + "/test/cats/cat.1080.jpg"
==> YOUR CODE HERE

In [ ]:
# TODO: Performance inference on this test image: dog.1132.jpg
==> YOUR CODE HERE